# Misc 1: Understanding Permutation Importance

This notebook walks through **permutation importance** step by step using the Iris dataset and a Random Forest classifier. We'll see exactly what happens to predictions when we shuffle (permute) one feature — first for a *genuinely important* feature, then for a *bogus* feature we invent ourselves that has no relationship to the target.

All the cells should run without any modifications, but it's recommended that you run one cell at a time as you read through this. 

You can open this notebook in either Google Colab or Kaggle:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sgeinitz/CS3120/blob/main/misc01_permutation_importance.ipynb)

[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/sgeinitz/CS3120/blob/main/misc01_permutation_importance.ipynb)

## Step-by-step recap

The big question permutation importance answers: **"How much does my trained model rely on this particular feature?"**

The trick: take a feature column and *scramble its values across rows*. The column still has the same numbers (same mean, variance, distribution), but those numbers are no longer matched to the right row. Then ask the same trained model to predict again. If it does much worse, the feature was important. If it does just as well, the model wasn't really using that feature.

### The procedure

1. **Train** your model on the training data.
2. **Measure baseline performance** on a held-out test set (e.g., accuracy = 0.95). Call this $S_{\text{baseline}}$.
3. **For each feature $j$:**
   - Copy the test data.
   - **Randomly shuffle column $j$** in that copy. Other columns untouched.
   - **Use the same already-trained model** to predict on this corrupted copy. *Do not retrain!*
   - Measure the new score $S_j$.
   - **Importance of feature $j$** $= S_{\text{baseline}} - S_j$.
4. (Optional but recommended) Repeat the shuffle several times and average — the result depends on the particular random shuffle.

### Why this works

- Shuffling preserves the column's *marginal* distribution (same values, same histogram) but breaks its relationship with the target.
- A feature the model genuinely depends on → shuffling destroys useful signal → predictions degrade → big importance.
- A feature the model ignores → shuffling changes nothing meaningful → predictions stay similar → near-zero importance.

### Common confusions

- **"Do we retrain after shuffling?"** No. We test what *this* trained model relies on.
- **"Why shuffle instead of dropping the column?"** Dropping changes the input shape and forces a retrain — then we'd be measuring a *different* model. Shuffling keeps the same model and neutralizes one feature.
- **"Why does the number change if I run it twice?"** The shuffle is random. Average several runs for stability.

## Setup: imports and reproducibility

We import the libraries we'll need and seed a random number generator so the results are reproducible from run to run.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

RNG = np.random.default_rng(seed=42)
np.random.seed(42)
pd.set_option('display.float_format', lambda v: f'{v:.3f}')

## Part 1 — A genuinely important feature

### Step 1: Load the data and train a model

We use Iris (predict flower species from 4 measurements) and a Random Forest classifier. We split the data into a train set (60%) and a test set (40%), fit the model on the training set, and record the baseline test accuracy. This baseline is the number every "importance" measurement below will be compared against.

In [ ]:
iris = load_iris()
feature_names = iris.feature_names
X = pd.DataFrame(iris.data, columns=feature_names)
y = pd.Series(iris.target, name='species')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

baseline_preds = model.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_preds)

print(f"Test set size:     {len(X_test)} rows")
print(f"Baseline accuracy: {baseline_accuracy:.4f}")

### Step 2: Pick a feature and look at the original predictions

We pick **`petal length (cm)`** — for Iris this feature is known to be highly informative; the three species have visibly different petal lengths. The table below shows the first 10 test rows with their true labels and the model's predictions on the *unshuffled* (original) data, plus a flag showing whether each prediction was correct.

In [ ]:
target_feature = 'petal length (cm)'

original_view = X_test.copy()
original_view['true_label']  = y_test.values
original_view['prediction']  = baseline_preds
original_view['correct?']    = (y_test.values == baseline_preds)

print(f"Predictions using ORIGINAL '{target_feature}' (first 10 test rows):")
original_view.head(10)

### Step 3: Shuffle that one column and predict again

Now we make a copy of the test set and randomly permute *only* the `petal length (cm)` column. Every other column stays exactly the same. We then call `.predict` with the **same trained model** — no retraining.

Look at the table below: the petal length values are still real Iris values (we only reordered them), but they're now attached to the wrong rows. Many predictions will change as a result.

In [ ]:
X_test_shuffled = X_test.copy()
X_test_shuffled[target_feature] = RNG.permutation(X_test_shuffled[target_feature].values)

shuffled_preds    = model.predict(X_test_shuffled)
shuffled_accuracy = accuracy_score(y_test, shuffled_preds)

shuffled_view = X_test_shuffled.copy()
shuffled_view['true_label'] = y_test.values
shuffled_view['prediction'] = shuffled_preds
shuffled_view['correct?']   = (y_test.values == shuffled_preds)

print(f"Predictions using SHUFFLED '{target_feature}' (first 10 test rows):")
shuffled_view.head(10)

### Step 4: Compare the two side by side

This is the heart of the technique. The same rows and the same model — only `petal length (cm)` was permuted. The `flipped?` column flags rows where the prediction changed between the original and shuffled runs.

Pay attention to two numbers in the printout: the accuracy drop, and how many predictions flipped. For an important feature, both should be substantial.

In [ ]:
comparison = pd.DataFrame({
    'true_label':                    y_test.values,
    f'{target_feature} (original)':  X_test[target_feature].values,
    'pred (original)':               baseline_preds,
    f'{target_feature} (shuffled)':  X_test_shuffled[target_feature].values,
    'pred (shuffled)':               shuffled_preds,
}, index=X_test.index)
comparison['flipped?'] = comparison['pred (original)'] != comparison['pred (shuffled)']

print(f"Baseline accuracy:   {baseline_accuracy:.4f}")
print(f"Shuffled accuracy:   {shuffled_accuracy:.4f}")
print(f"DROP (= importance): {baseline_accuracy - shuffled_accuracy:.4f}")
print(f"Predictions that changed: {comparison['flipped?'].sum()} out of {len(comparison)}")
print()
print('First 15 rows:')
comparison.head(15)

### Step 5: That drop *is* the permutation importance

$$
\text{Importance}(\text{petal length}) = S_{\text{baseline}} - S_{\text{shuffled}}
$$

Because petal length really matters for telling Iris species apart, scrambling it makes the model substantially worse — many predictions flip and accuracy drops sharply.

A single shuffle gives a noisy estimate (you'd get a slightly different number with a different random seed). The standard fix is to **repeat the shuffle several times and report the mean and standard deviation**. The function below does exactly that, and we'll reuse it later for the bogus feature.

In [ ]:
def permutation_importance_for(model, X_test, y_test, feature, baseline, n_repeats=10, seed=42):
    """Mean and std of the drop in accuracy across n_repeats shuffles of one column."""
    rng = np.random.default_rng(seed)
    drops = []
    for _ in range(n_repeats):
        X_perm = X_test.copy()
        X_perm[feature] = rng.permutation(X_perm[feature].values)
        score = accuracy_score(y_test, model.predict(X_perm))
        drops.append(baseline - score)
    return float(np.mean(drops)), float(np.std(drops))

mean_imp, std_imp = permutation_importance_for(
    model, X_test, y_test, target_feature, baseline_accuracy, n_repeats=30
)
print(f"Permutation importance for '{target_feature}': {mean_imp:.4f}  (std {std_imp:.4f}, over 30 shuffles)")

## Part 2 — A bogus feature with no relationship to the target

Now we deliberately invent a feature that's pure random noise — values drawn from a standard normal distribution that have **nothing to do with the species label**. We then add this column to our data, retrain the Random Forest on the augmented features, and run the same procedure as before.

If permutation importance works correctly, shuffling this feature should change almost nothing — its importance should hover around zero. That's the result we want to demonstrate to convince students the method is actually doing what it claims.

In [ ]:
# Pure random noise. No relationship to species at all.
X_with_bogus = X.copy()
X_with_bogus['bogus_noise'] = RNG.normal(loc=0, scale=1, size=len(X))

X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X_with_bogus, y, test_size=0.4, random_state=42, stratify=y
)

model_b = RandomForestClassifier(n_estimators=100, random_state=42)
model_b.fit(X_train_b, y_train_b)

baseline_preds_b    = model_b.predict(X_test_b)
baseline_accuracy_b = accuracy_score(y_test_b, baseline_preds_b)

print(f"Baseline accuracy WITH bogus feature included: {baseline_accuracy_b:.4f}")
print(f"(Compare to {baseline_accuracy:.4f} without it — adding noise barely changes anything.)")

### Step 2 (bogus): predictions with the original bogus feature

Just like in Part 1, we first record what the model predicts on the test set with the bogus column in its original (unshuffled) state. These predictions will be our reference point for the shuffled version.

In [ ]:
bogus_feature = 'bogus_noise'

orig_view_b = X_test_b.copy()
orig_view_b['true_label'] = y_test_b.values
orig_view_b['prediction'] = baseline_preds_b
orig_view_b['correct?']   = (y_test_b.values == baseline_preds_b)

print(f"Predictions using ORIGINAL '{bogus_feature}' (first 10 test rows):")
orig_view_b.head(10)

### Step 3 (bogus): shuffle the bogus feature and predict again

Same exact procedure as before: copy the test set, permute *only* the `bogus_noise` column, predict with the same trained model. The only difference is which column we're shuffling.

In [ ]:
X_test_b_shuffled = X_test_b.copy()
X_test_b_shuffled[bogus_feature] = RNG.permutation(X_test_b_shuffled[bogus_feature].values)

shuffled_preds_b    = model_b.predict(X_test_b_shuffled)
shuffled_accuracy_b = accuracy_score(y_test_b, shuffled_preds_b)

shuf_view_b = X_test_b_shuffled.copy()
shuf_view_b['true_label'] = y_test_b.values
shuf_view_b['prediction'] = shuffled_preds_b
shuf_view_b['correct?']   = (y_test_b.values == shuffled_preds_b)

print(f"Predictions using SHUFFLED '{bogus_feature}' (first 10 test rows):")
shuf_view_b.head(10)

### Step 4 (bogus): compare side by side

Same comparison table as before. The key thing to look at: the `flipped?` column should be almost entirely `False`, the accuracy drop should be tiny, and the original/shuffled predictions should look nearly identical. The model never relied on this column, so corrupting it doesn't hurt.

In [ ]:
comparison_b = pd.DataFrame({
    'true_label':                   y_test_b.values,
    f'{bogus_feature} (original)':  X_test_b[bogus_feature].values,
    'pred (original)':              baseline_preds_b,
    f'{bogus_feature} (shuffled)':  X_test_b_shuffled[bogus_feature].values,
    'pred (shuffled)':              shuffled_preds_b,
}, index=X_test_b.index)
comparison_b['flipped?'] = comparison_b['pred (original)'] != comparison_b['pred (shuffled)']

print(f"Baseline accuracy:   {baseline_accuracy_b:.4f}")
print(f"Shuffled accuracy:   {shuffled_accuracy_b:.4f}")
print(f"DROP (= importance): {baseline_accuracy_b - shuffled_accuracy_b:.4f}")
print(f"Predictions that changed: {comparison_b['flipped?'].sum()} out of {len(comparison_b)}")
print()
print('First 15 rows:')
comparison_b.head(15)

### Step 5 (bogus): importance of the bogus feature

We expect this to hover around 0. It can even come out **slightly negative** on a single run due to randomness — that's a useful sanity check, not a bug: a negative importance just means that on this particular shuffle the corrupted column happened to give marginally better predictions, which is exactly what you'd expect for a column the model isn't really using.

The point is: nowhere near the importance of `petal length (cm)`.

In [ ]:
mean_imp_b, std_imp_b = permutation_importance_for(
    model_b, X_test_b, y_test_b, bogus_feature, baseline_accuracy_b, n_repeats=30
)
print(f"Permutation importance for '{bogus_feature}':       {mean_imp_b:+.4f}  (std {std_imp_b:.4f})")
print(f"Permutation importance for '{target_feature}':  {mean_imp:+.4f}  (std {std_imp:.4f})")

## Part 3 — All features at a glance

Finally, we compute permutation importance for *every* column in the model that includes the bogus feature, and plot them as a horizontal bar chart with error bars (one standard deviation across 30 shuffles).

The bogus column should clearly be smallest, sitting at or very close to zero. The truly informative features (especially the petal measurements) should tower above it.

In [ ]:
results = []
for feat in X_test_b.columns:
    m, s = permutation_importance_for(
        model_b, X_test_b, y_test_b, feat, baseline_accuracy_b, n_repeats=30
    )
    results.append({'feature': feat, 'importance_mean': m, 'importance_std': s})

importance_df = pd.DataFrame(results).sort_values('importance_mean')
print(importance_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#d62728' if f == 'bogus_noise' else '#1f77b4' for f in importance_df['feature']]
ax.barh(importance_df['feature'], importance_df['importance_mean'],
        xerr=importance_df['importance_std'], color=colors, capsize=4)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Mean drop in accuracy when feature is shuffled (= permutation importance)')
ax.set_title('Permutation importance per feature  (red = bogus feature)')
plt.tight_layout()
plt.show()

## Takeaways

- Permutation importance is **model-agnostic** and runs *after* training is done.
- We measure how much performance drops when one feature's column is randomly scrambled.
- A genuinely useful feature → big drop → high importance.
- A useless / bogus feature → tiny drop (or even slightly negative) → near-zero importance.
- We do **not** retrain. The same model is being asked to predict on data where one column is now noise.
- Because the shuffle is random, repeat it (e.g. 10–30 times) and report the mean and standard deviation.

### A useful diagnostic trick

Adding a known bogus feature (like we did) and computing its permutation importance gives you a **noise floor**: any real feature whose importance isn't clearly above the bogus feature's importance probably isn't doing meaningful work either.